# ECon3D TRELLIS - Precompiled GPU Setup

Select a T4 GPU runtime, then choose Runtime > Run all. No installation on your PC is required. Uses binary wheels for Python 3.10 / PyTorch 2.4.0 / CUDA 12.1. If a compatible wheel is unavailable, setup stops instead of compiling from source.

Starts a persistent TRELLIS API and a verified Cloudflare HTTPS tunnel. API requests require a per-session Bearer token. The online address registry is not configured yet. This is not a completed customer auto-connect release.

In [ ]:
SOURCES={'bootstrap.py': '"""Colab-only binary installation. No source compilation fallback."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport urllib.request\n\nROOT = Path(\'/content/ECon3D_Colab_Server\')\nROOT.mkdir(exist_ok=True)\nPY = str(ROOT/\'venv/bin/python\')\nREPO = ROOT/\'TRELLIS\'\nREV = \'977dc2e6d8a2c47f2e559a1d4abce5720d1584c8\'\nenv = os.environ.copy()\nenv.update(ATTN_BACKEND=\'xformers\', SPCONV_ALGO=\'native\',\n           HF_HOME=str(ROOT/\'cache/hf\'), TORCH_HOME=str(ROOT/\'cache/torch\'),\n           PYTHONPATH=str(REPO)+\':\'+str(ROOT/\'utils3d\'))\nstarted = time.perf_counter()\nstages = {}\n\n\ndef command(args, cwd=None):\n    proc = subprocess.Popen(list(map(str, args)), cwd=cwd, env=env,\n                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n    for line in proc.stdout:\n        print(line, end=\'\', flush=True)\n    if proc.wait(): raise subprocess.CalledProcessError(proc.returncode, args)\n\n\ndef pip(*args):\n    command([sys.executable, \'-m\', \'uv\', \'pip\', \'install\', \'--python\', PY,\n             \'--only-binary=:all:\', *args])\n\n\ndef healthy():\n    if not Path(PY).exists() or not REPO.exists(): return False\n    result = subprocess.run([PY, \'-c\',\n        \'import torch,xformers,spconv,kaolin,nvdiffrast.torch; \'\n        \'from trellis.pipelines import TrellisImageTo3DPipeline; \'\n        \'assert torch.cuda.is_available(); assert torch.__version__.startswith("2.4.0")\'],\n        env=env, cwd=REPO, capture_output=True, text=True)\n    print(result.stdout, flush=True)\n    if result.returncode: print(result.stderr, flush=True)\n    return result.returncode == 0\n\n\ndef setup():\n    command([\'nvidia-smi\', \'--query-gpu=name,memory.total\', \'--format=csv\'])\n    digest = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n    marker = ROOT/\'install_ready.json\'\n    saved = json.loads(marker.read_text()) if marker.exists() else {}\n    if saved.get(\'recipe\') == digest and healthy():\n        print(\'INSTALL_REUSED: same healthy runtime; no download/install\', flush=True)\n        return\n    t = time.perf_counter()\n    command([sys.executable, \'-m\', \'pip\', \'install\', \'--only-binary=:all:\', \'uv==0.12.13\'])\n    if not Path(PY).exists():\n        command([sys.executable, \'-m\', \'uv\', \'venv\', \'--python\', \'3.10\', ROOT/\'venv\'])\n    pip(\'torch==2.4.0\', \'torchvision==0.19.0\', \'xformers==0.0.27.post2\',\n        \'--index-url\', \'https://download.pytorch.org/whl/cu121\')\n    stages[\'python_torch_seconds\'] = time.perf_counter()-t\n    t = time.perf_counter()\n    pip(\'numpy==1.26.4\', \'pillow==10.4.0\', \'imageio==2.36.1\', \'imageio-ffmpeg==0.5.1\',\n        \'tqdm==4.67.1\', \'easydict==1.13\', \'opencv-python-headless==4.10.0.84\',\n        \'scipy==1.14.1\', \'rembg==2.0.60\', \'onnxruntime==1.20.1\', \'trimesh==4.5.3\',\n        \'xatlas==0.0.9\', \'pyvista==0.44.2\', \'pymeshfix==0.17.0\', \'igraph==0.11.8\',\n        \'spconv-cu120==2.3.6\', \'transformers==4.46.3\', \'huggingface-hub<1\',\n        \'fastapi\', \'uvicorn\', \'requests\', \'einops\', \'safetensors\', \'plyfile==1.1.3\')\n    pip(\'kaolin==0.17.0\', \'--find-links\',\n        \'https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html\')\n    for wheel in (\'diff_gaussian_rasterization-0.0.0-cp310-cp310-linux_x86_64.whl\',\n                  \'nvdiffrast-0.3.3-cp310-cp310-linux_x86_64.whl\'):\n        pip(\'https://huggingface.co/spaces/microsoft/TRELLIS/resolve/\'+REV+\'/wheels/\'+wheel)\n    stages[\'binary_dependencies_seconds\'] = time.perf_counter()-t\n    t = time.perf_counter()\n    # Source-only Python modules; never compile CUDA extensions in a customer runtime.\n    command([PY, \'-c\', \'from huggingface_hub import snapshot_download; \'\n        f\'snapshot_download("microsoft/TRELLIS",repo_type="space",revision="{REV}",\'\n        f\'local_dir={str(REPO)!r},allow_patterns=["trellis/**"],ignore_patterns=["*.pyc"])\'])\n    if not (ROOT/\'utils3d\').exists():\n        command([\'git\', \'clone\', \'https://github.com/EasternJournalist/utils3d.git\', ROOT/\'utils3d\'])\n    command([\'git\', \'checkout\', \'9a4eb15e4021b67b12c460c7057d642626897ec8\'], cwd=ROOT/\'utils3d\')\n    stages[\'python_sources_seconds\'] = time.perf_counter()-t\n    if not healthy(): raise RuntimeError(\'Binary environment import/GPU check failed. No source-build fallback.\')\n    report = dict(status=\'INSTALL_READY\', recipe=digest, source_revision=REV,\n                  stages=stages, total_seconds=time.perf_counter()-started,\n                  model_ready=False, connection_ready=False)\n    marker.write_text(json.dumps(report, indent=2))\n    print(json.dumps(report, indent=2), flush=True)\n\n\nif __name__ == \'__main__\': setup()\n', 'service.py': '"""Persistent TRELLIS model, authenticated geometry endpoint and honest health."""\nimport base64\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport secrets\nimport threading\nimport time\nimport zipfile\n\nfrom fastapi import FastAPI, HTTPException, Request\nfrom fastapi.responses import Response\nfrom starlette.concurrency import run_in_threadpool\nimport numpy as np\nfrom PIL import Image, ImageOps\nimport torch\nimport trimesh\nfrom trellis.pipelines import TrellisImageTo3DPipeline\n\napp = FastAPI()\nstate = {\'status\':\'LOADING\', \'model\':\'TRELLIS-image-large\', \'model_ready\':False,\n         \'inference_verified\':False}\nlock = threading.Lock()\npipeline = None\ntoken = os.environ.get(\'ECON3D_API_TOKEN\', \'\')\nif not token: raise RuntimeError(\'Missing per-session API token.\')\n\n\ndef authorize(request):\n    if not secrets.compare_digest(request.headers.get(\'Authorization\', \'\'), \'Bearer \'+token):\n        raise HTTPException(401, \'Unauthorized\')\n\n\ndef load():\n    global pipeline\n    start = time.perf_counter()\n    try:\n        pipeline = TrellisImageTo3DPipeline.from_pretrained(\'microsoft/TRELLIS-image-large\')\n        pipeline.cuda()\n        state.update(status=\'READY\', model_ready=True, gpu=torch.cuda.get_device_name(),\n                     model_load_seconds=time.perf_counter()-start)\n    except Exception as error:\n        state.update(status=\'FAIL\', error=str(error))\n    Path(\'/content/ECon3D_Colab_Server/model_status.json\').write_text(json.dumps(state, indent=2))\n    print(json.dumps(state), flush=True)\n\n\n@app.on_event(\'startup\')\ndef startup(): threading.Thread(target=load, daemon=True).start()\n\n\n@app.get(\'/\')\n@app.get(\'/health\')\ndef health(request: Request):\n    authorize(request)\n    return dict(state)\n\n\n@app.post(\'/generate-3d\')\n@app.post(\'/generate\')\nasync def generate(request: Request):\n    authorize(request)\n    if not state[\'model_ready\']: raise HTTPException(503, \'Model not ready\')\n    body = bytearray()\n    async for chunk in request.stream():\n        body.extend(chunk)\n        if len(body) > 24*1024*1024: raise HTTPException(413, \'Input too large\')\n    try:\n        payload = json.loads(body)\n        raw_images = payload.get(\'images\')\n        if raw_images is None: raw_images = [payload[\'image_b64\']]\n        seed = int(payload.get(\'seed\', 1))\n        texture_size = int(payload.get(\'texture_size\', 1024 if request.url.path == \'/generate-3d\' else 0))\n        if texture_size not in (0, 512, 1024, 2048):\n            raise ValueError(\'texture_size must be 0, 512, 1024 or 2048\')\n        if not 0 <= seed < 2**32: raise ValueError(\'Seed out of range\')\n        if not isinstance(raw_images, list) or not 1 <= len(raw_images) <= 6:\n            raise ValueError(\'Use 1 to 6 photographs of the same object.\')\n        images = []\n        for value in raw_images:\n            image = Image.open(io.BytesIO(base64.b64decode(value.split(\',\')[-1], validate=True)))\n            if image.width*image.height > 20_000_000: raise ValueError(\'Image exceeds 20 megapixels\')\n            images.append(ImageOps.exif_transpose(image).convert(\'RGBA\'))\n    except Exception as error:\n        raise HTTPException(400, str(error))\n    if not lock.acquire(blocking=False): raise HTTPException(409, \'Generation in progress\')\n    return await run_in_threadpool(generate_output, images, seed, texture_size, request.url.path)\n\n\ndef generate_output(images, seed, texture_size, endpoint):\n    try:\n        start = time.perf_counter()\n        processed = [pipeline.preprocess_image(image) for image in images]\n        options = dict(seed=seed, formats=[\'mesh\', \'gaussian\'] if texture_size else [\'mesh\'], preprocess_image=False,\n                       sparse_structure_sampler_params={\'steps\':12, \'cfg_strength\':7.5},\n                       slat_sampler_params={\'steps\':12, \'cfg_strength\':3.0})\n        with torch.inference_mode():\n            result = pipeline.run(processed[0], **options) if len(processed)==1 else pipeline.run_multi_image(\n                processed, mode=\'multidiffusion\', **options)\n        raw = result[\'mesh\'][0]\n        xyz = raw.vertices.detach().cpu().numpy()\n        faces = raw.faces.detach().cpu().numpy()\n        if not len(xyz) or not len(faces) or not np.isfinite(xyz).all():\n            raise RuntimeError(\'Invalid output geometry\')\n        mesh = trimesh.Trimesh(xyz, faces, process=False)\n        if texture_size:\n            from trellis.utils import postprocessing_utils\n            textured = postprocessing_utils.to_glb(result[\'gaussian\'][0], raw, texture_size=texture_size)\n            glb_bytes = textured.export(file_type=\'glb\')\n        else:\n            glb_bytes = mesh.export(file_type=\'glb\')\n        report = dict(status=\'GENERATED\', vertices=len(xyz), faces=len(faces),\n                      input_count=len(images), elapsed_seconds=time.perf_counter()-start,\n                      metric_calibrated=False, manual_adjustments=False)\n        archive = io.BytesIO()\n        with zipfile.ZipFile(archive, \'w\', zipfile.ZIP_DEFLATED) as z:\n            z.writestr(\'model.glb\', glb_bytes)\n            z.writestr(\'model.obj\', mesh.export(file_type=\'obj\'))\n            points = io.BytesIO(); np.savez_compressed(points, xyz=xyz, faces=faces)\n            z.writestr(\'vertices.npz\', points.getvalue())\n            z.writestr(\'result.json\', json.dumps(report))\n            for i, image in enumerate(processed):\n                png = io.BytesIO(); image.save(png, format=\'PNG\')\n                z.writestr(f\'processed_{i}.png\', png.getvalue())\n        state[\'inference_verified\'] = True\n        if endpoint == \'/generate-3d\':\n            return dict(ok=True, glb_base64=base64.b64encode(glb_bytes).decode(),\n                        file_size=len(glb_bytes), bytes_length=len(glb_bytes),\n                        geometry_only=not bool(texture_size), **report)\n        return Response(archive.getvalue(), media_type=\'application/zip\')\n    finally:\n        lock.release()\n', 'tunnel.py': '"""Verified Cloudflare binary, ready-only HTTPS publication. No public API key."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport subprocess\nimport time\nimport urllib.request\n\nVERSION = \'2026.9.1\'\nSHA256 = \'03f1f25d1cc93b9ad6c60569d44060bc4f17ed97075760ed8cfca4b12dcd68cc\'\n\n\ndef read_health(url, token):\n    request = urllib.request.Request(url.rstrip(\'/\')+\'/health\',\n                                     headers={\'Authorization\':\'Bearer \'+token})\n    with urllib.request.urlopen(request, timeout=12) as response:\n        data = json.load(response)\n    if data.get(\'status\') != \'READY\' or data.get(\'model_ready\') is not True:\n        raise RuntimeError(\'Model is not ready.\')\n    return data\n\n\ndef start(root, token):\n    root = Path(root)\n    read_health(\'http://127.0.0.1:8000\', token)\n    binary = root/\'cloudflared\'\n    if not binary.exists() or hashlib.sha256(binary.read_bytes()).hexdigest() != SHA256:\n        url = f\'https://github.com/cloudflare/cloudflared/releases/download/{VERSION}/cloudflared-linux-amd64\'\n        with urllib.request.urlopen(url, timeout=60) as response: data = response.read()\n        if hashlib.sha256(data).hexdigest() != SHA256:\n            raise RuntimeError(\'Cloudflare executable checksum mismatch\')\n        binary.write_bytes(data)\n        binary.chmod(0o700)\n    connection = root/\'connection.json\'\n    if connection.exists():\n        old = json.loads(connection.read_text())\n        if old.get(\'expires_at\', 0) > time.time():\n            try:\n                read_health(old[\'url\'], token)\n                print(\'HTTPS_REUSED: \'+old[\'url\'], flush=True)\n                return old\n            except Exception: pass\n    log_path = root/\'tunnel.log\'\n    with log_path.open(\'w\') as log:\n        process = subprocess.Popen([str(binary), \'tunnel\', \'--url\', \'http://127.0.0.1:8000\',\n                                    \'--no-autoupdate\', \'--protocol\', \'http2\'],\n                                   stdout=log, stderr=subprocess.STDOUT)\n    deadline = time.monotonic()+120\n    last_error = \'Waiting for Cloudflare URL\'\n    while time.monotonic() < deadline:\n        if process.poll() is not None: raise RuntimeError(\'Cloudflare process exited; see tunnel.log\')\n        match = re.search(r\'https://[a-z0-9-]+\\.trycloudflare\\.com\', log_path.read_text())\n        if match:\n            try:\n                status = read_health(match.group(), token)\n                result = dict(url=match.group(), status=\'HTTPS_READY\',\n                              model=status[\'model\'], created_at=time.time(),\n                              expires_at=time.time()+3600, pid=process.pid,\n                              auto_sync=False)\n                connection.write_text(json.dumps(result, indent=2))\n                print(\'HTTPS_READY: \'+result[\'url\'], flush=True)\n                return result\n            except Exception as error: last_error = str(error)\n        time.sleep(2)\n    process.terminate()\n    raise RuntimeError(\'HTTPS readiness failed: \'+last_error)\n'}

import os, subprocess, pathlib, secrets, time, json, urllib.request
ROOT=pathlib.Path('/content/ECon3D_Colab_Server')
ROOT.mkdir(exist_ok=True)
for name, source in SOURCES.items(): (ROOT/name).write_text(source,encoding='utf8')
import runpy
setup_scope=runpy.run_path(str(ROOT/'bootstrap.py'))
setup_scope['setup']()
PY=setup_scope['PY']; env=setup_scope['env']


In [ ]:

# Model stays loaded: subsequent requests do not reinstall or reload it.
import socket
with socket.socket() as probe:
    occupied=probe.connect_ex(('127.0.0.1',8000))==0
token_file=ROOT/'session_token'
if occupied and not token_file.exists():
    raise RuntimeError('Port 8000 belongs to an unknown process; not stopped.')
API_TOKEN=token_file.read_text() if occupied else secrets.token_urlsafe(32)
if not occupied:
    token_file.write_text(API_TOKEN)
    token_file.chmod(0o600)
env['ECON3D_API_TOKEN']=API_TOKEN
server=None
if not occupied:
    server_log=(ROOT/'server.log').open('w')
    server=subprocess.Popen([PY,'-m','uvicorn','service:app','--host','127.0.0.1','--port','8000'],
        cwd=ROOT,env=env,stdout=server_log,stderr=subprocess.STDOUT)
deadline=time.monotonic()+1200
last=''
while time.monotonic()<deadline:
    if server is not None and server.poll() is not None:
        print((ROOT/'server.log').read_text()[-10000:])
        raise RuntimeError('Server startup failed.')
    try:
        req=urllib.request.Request('http://127.0.0.1:8000/health',headers={'Authorization':'Bearer '+API_TOKEN})
        with urllib.request.urlopen(req,timeout=5) as response: status=json.load(response)
        if status['status']=='FAIL': raise RuntimeError(status.get('error'))
        if status['model_ready']:
            print('MODEL_READY: model loaded; image inference not yet tested.')
            print(json.dumps(status,indent=2)); break
    except (OSError,ValueError): pass
    if int(time.monotonic())//30 != last:
        last=int(time.monotonic())//30
        print('Model loading. Log:', (ROOT/'server.log').read_text()[-400:],flush=True)
    time.sleep(2)
else: raise TimeoutError('Model loading timed out; see server.log.')
print('MODEL_READY. Starting authenticated HTTPS tunnel next.')


In [ ]:

tunnel_scope=runpy.run_path(str(ROOT/'tunnel.py'))
connection=tunnel_scope['start'](ROOT,API_TOKEN)
print('POST '+connection['url']+'/generate-3d')
print('Requests require the per-session Bearer token; it is not published in this notebook.')
print('AUTO_SYNC_NOT_CONFIGURED: the online address registry is not provisioned yet.')
